In [8]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Cargar los puntos actualizados (sin el 10 de julio)
df_puntos = spark.read.table("stg_puntos_gpx")

# 2. Recalcular distancias y desniveles
ventana_global = Window.orderBy("fecha_hora")

R = 6371.0
lat1 = F.radians(F.lag("latitud").over(ventana_global))
lon1 = F.radians(F.lag("longitud").over(ventana_global))
lat2 = F.radians(F.col("latitud"))
lon2 = F.radians(F.col("longitud"))

dlat = lat2 - lat1
dlon = lon2 - lon1
a = F.sin(dlat / 2)**2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2)**2
c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
distancia_tramo = R * c

diferencia_altitud = F.col("elevacion") - F.lag("elevacion").over(ventana_global)
desnivel_positivo = F.when(diferencia_altitud > 0, diferencia_altitud).otherwise(0)

tiempo_seg = F.col("fecha_hora").cast("long") - F.lag("fecha_hora").over(ventana_global).cast("long")
tiempo_movimiento_seg = F.when((tiempo_seg > 0) & (tiempo_seg < 900), tiempo_seg).otherwise(0)

df_preparado = df_puntos \
    .withColumn("fecha_diaria", F.to_date("fecha_hora")) \
    .withColumn("dist_km", F.coalesce(distancia_tramo, F.lit(0.0))) \
    .withColumn("desnivel_pos_m", F.coalesce(desnivel_positivo, F.lit(0.0))) \
    .withColumn("tiempo_mov_seg", F.coalesce(tiempo_movimiento_seg, F.lit(0)))

# 3. Agrupar por FECHA DIARIA
df_gold_fechas = df_preparado.groupBy("fecha_diaria").agg(
    F.min("fecha_hora").alias("hora_inicio_jornada"),
    F.max("fecha_hora").alias("hora_fin_jornada"),
    F.round(F.sum("dist_km"), 2).alias("distancia_total_km"),
    F.round(F.min("elevacion"), 1).alias("altitud_min_m"),
    F.round(F.max("elevacion"), 1).alias("altitud_max_m"),
    F.round(F.sum("desnivel_pos_m"), 1).alias("desnivel_acumulado_m"),
    F.round(F.sum("tiempo_mov_seg") / 3600, 2).alias("duracion_movimiento_horas"),
    F.count("*").alias("total_puntos_registrados")
).orderBy("fecha_diaria")

# 4. Velocidad media
df_gold_fechas = df_gold_fechas \
    .withColumn("velocidad_media_kmh", 
                F.when(F.col("duracion_movimiento_horas") > 0, 
                       F.round(F.col("distancia_total_km") / F.col("duracion_movimiento_horas"), 1))
                .otherwise(0.0))

# 5. Sobrescribir Gold y refrescar catálogo
df_gold_fechas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("gold_rutas_resumen")

spark.catalog.refreshTable("stg_puntos_gpx")
spark.catalog.refreshTable("gold_rutas_resumen")

print("🏆 ¡Tabla Gold limpia de la fecha 10 de julio!")
display(spark.read.table("gold_rutas_resumen"))

StatementMeta(, e19628f8-5812-484b-8f2b-c6bd368faece, 10, Finished, Available, Finished, False)

🏆 ¡Tabla Gold limpia de la fecha 10 de julio!


SynapseWidget(Synapse.DataFrame, 7be0a7f0-2988-4f66-9b72-c8ccbde50798)

In [7]:
from pyspark.sql import functions as F

# Cargar Silver y filtrar excluyendo la fecha 2026-07-16
df_silver_sin_10jul = spark.read.table("stg_puntos_gpx") \
    .filter(F.to_date("fecha_hora") != "2026-07-16")

# Sobrescribir la tabla Silver
df_silver_sin_10jul.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("stg_puntos_gpx")

print("✅ Datos del 16 de julio de 2026 eliminados correctamente de 'stg_puntos_gpx'.")

StatementMeta(, e19628f8-5812-484b-8f2b-c6bd368faece, 9, Finished, Available, Finished, False)

✅ Datos del 16 de julio de 2026 eliminados correctamente de 'stg_puntos_gpx'.


In [6]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Cargar los puntos actualizados (sin el 10 de julio)
df_puntos = spark.read.table("stg_puntos_gpx")

# 2. Recalcular distancias y desniveles
ventana_global = Window.orderBy("fecha_hora")

R = 6371.0
lat1 = F.radians(F.lag("latitud").over(ventana_global))
lon1 = F.radians(F.lag("longitud").over(ventana_global))
lat2 = F.radians(F.col("latitud"))
lon2 = F.radians(F.col("longitud"))

dlat = lat2 - lat1
dlon = lon2 - lon1
a = F.sin(dlat / 2)**2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2)**2
c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
distancia_tramo = R * c

diferencia_altitud = F.col("elevacion") - F.lag("elevacion").over(ventana_global)
desnivel_positivo = F.when(diferencia_altitud > 0, diferencia_altitud).otherwise(0)

tiempo_seg = F.col("fecha_hora").cast("long") - F.lag("fecha_hora").over(ventana_global).cast("long")
tiempo_movimiento_seg = F.when((tiempo_seg > 0) & (tiempo_seg < 900), tiempo_seg).otherwise(0)

df_preparado = df_puntos \
    .withColumn("fecha_diaria", F.to_date("fecha_hora")) \
    .withColumn("dist_km", F.coalesce(distancia_tramo, F.lit(0.0))) \
    .withColumn("desnivel_pos_m", F.coalesce(desnivel_positivo, F.lit(0.0))) \
    .withColumn("tiempo_mov_seg", F.coalesce(tiempo_movimiento_seg, F.lit(0)))

# 3. Agrupar por FECHA DIARIA
df_gold_fechas = df_preparado.groupBy("fecha_diaria").agg(
    F.min("fecha_hora").alias("hora_inicio_jornada"),
    F.max("fecha_hora").alias("hora_fin_jornada"),
    F.round(F.sum("dist_km"), 2).alias("distancia_total_km"),
    F.round(F.min("elevacion"), 1).alias("altitud_min_m"),
    F.round(F.max("elevacion"), 1).alias("altitud_max_m"),
    F.round(F.sum("desnivel_pos_m"), 1).alias("desnivel_acumulado_m"),
    F.round(F.sum("tiempo_mov_seg") / 3600, 2).alias("duracion_movimiento_horas"),
    F.count("*").alias("total_puntos_registrados")
).orderBy("fecha_diaria")

# 4. Velocidad media
df_gold_fechas = df_gold_fechas \
    .withColumn("velocidad_media_kmh", 
                F.when(F.col("duracion_movimiento_horas") > 0, 
                       F.round(F.col("distancia_total_km") / F.col("duracion_movimiento_horas"), 1))
                .otherwise(0.0))

# 5. Sobrescribir Gold y refrescar catálogo
df_gold_fechas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("gold_rutas_resumen")

spark.catalog.refreshTable("stg_puntos_gpx")
spark.catalog.refreshTable("gold_rutas_resumen")

print("🏆 ¡Tabla Gold limpia de la fecha 10 de julio!")
display(spark.read.table("gold_rutas_resumen"))

StatementMeta(, e19628f8-5812-484b-8f2b-c6bd368faece, 8, Finished, Available, Finished, False)

🏆 ¡Tabla Gold limpia de la fecha 10 de julio!


SynapseWidget(Synapse.DataFrame, c69e69c9-836c-4600-b209-22d831828c35)

In [5]:
from pyspark.sql import functions as F

# Cargar Silver y filtrar excluyendo la fecha 2026-07-10
df_silver_sin_10jul = spark.read.table("stg_puntos_gpx") \
    .filter(F.to_date("fecha_hora") != "2026-07-10")

# Sobrescribir la tabla Silver
df_silver_sin_10jul.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("stg_puntos_gpx")

print("✅ Datos del 10 de julio de 2026 eliminados correctamente de 'stg_puntos_gpx'.")

StatementMeta(, e19628f8-5812-484b-8f2b-c6bd368faece, 7, Finished, Available, Finished, False)

✅ Datos del 10 de julio de 2026 eliminados correctamente de 'stg_puntos_gpx'.


In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Cargar los puntos limpios
df_puntos = spark.read.table("stg_puntos_gpx")

# 2. Calcular tiempo y distancia entre puntos consecutivos para medir tiempo en movimiento
ventana_global = Window.orderBy("fecha_hora")

tiempo_seg = F.col("fecha_hora").cast("long") - F.lag("fecha_hora").over(ventana_global).cast("long")

# Consideramos "tiempo en movimiento" si el salto entre puntos es menor a 15 minutos (900 seg)
# Si pasaron horas entre dos puntos (ej. durmiendo), ese salto no se suma al tiempo en marcha.
tiempo_movimiento_seg = F.when((tiempo_seg > 0) & (tiempo_seg < 900), tiempo_seg).otherwise(0)

R = 6371.0
lat1 = F.radians(F.lag("latitud").over(ventana_global))
lon1 = F.radians(F.lag("longitud").over(ventana_global))
lat2 = F.radians(F.col("latitud"))
lon2 = F.radians(F.col("longitud"))

dlat = lat2 - lat1
dlon = lon2 - lon1
a = F.sin(dlat / 2)**2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2)**2
c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
distancia_tramo = R * c

diferencia_altitud = F.col("elevacion") - F.lag("elevacion").over(ventana_global)
desnivel_positivo = F.when(diferencia_altitud > 0, diferencia_altitud).otherwise(0)

df_preparado = df_puntos \
    .withColumn("fecha_diaria", F.to_date("fecha_hora")) \
    .withColumn("dist_km", F.coalesce(distancia_tramo, F.lit(0.0))) \
    .withColumn("desnivel_pos_m", F.coalesce(desnivel_positivo, F.lit(0.0))) \
    .withColumn("tiempo_mov_seg", F.coalesce(tiempo_movimiento_seg, F.lit(0)))

# 3. Agrupar por fecha diaria
df_gold_fechas = df_preparado.groupBy("fecha_diaria").agg(
    F.min("fecha_hora").alias("hora_inicio_jornada"),
    F.max("fecha_hora").alias("hora_fin_jornada"),
    F.round(F.sum("dist_km"), 2).alias("distancia_total_km"),
    F.round(F.min("elevacion"), 1).alias("altitud_min_m"),
    F.round(F.max("elevacion"), 1).alias("altitud_max_m"),
    F.round(F.sum("desnivel_pos_m"), 1).alias("desnivel_acumulado_m"),
    F.round(F.sum("tiempo_mov_seg") / 3600, 2).alias("duracion_movimiento_horas"),
    F.count("*").alias("total_puntos_registrados")
).orderBy("fecha_diaria")

# 4. Calcular velocidad media REAL en movimiento
df_gold_fechas = df_gold_fechas \
    .withColumn("velocidad_media_kmh", 
                F.when(F.col("duracion_movimiento_horas") > 0, 
                       F.round(F.col("distancia_total_km") / F.col("duracion_movimiento_horas"), 1))
                .otherwise(0.0))

# 5. Sobrescribir Gold y refrescar
df_gold_fechas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("gold_rutas_resumen")

spark.catalog.refreshTable("gold_rutas_resumen")

print("🏆 ¡Tabla Gold corregida eliminando pauses nocturnos e inactividad!")
display(spark.read.table("gold_rutas_resumen").filter(F.col("fecha_diaria") == "2026-07-10"))

StatementMeta(, e19628f8-5812-484b-8f2b-c6bd368faece, 6, Finished, Available, Finished, False)

🏆 ¡Tabla Gold corregida eliminando pauses nocturnos e inactividad!


SynapseWidget(Synapse.DataFrame, 1ebd3d46-ebac-436a-9f5d-c905f98f902a)

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Cargar los puntos limpios (sin caminatas ni paradas)
df_puntos_limpios = spark.read.table("stg_puntos_gpx")

# 2. Re-calcular distancias y desniveles entre los puntos conservados
ventana_global = Window.orderBy("fecha_hora")

R = 6371.0  # Radio de la Tierra en km

lat1 = F.radians(F.lag("latitud").over(ventana_global))
lon1 = F.radians(F.lag("longitud").over(ventana_global))
lat2 = F.radians(F.col("latitud"))
lon2 = F.radians(F.col("longitud"))

dlat = lat2 - lat1
dlon = lon2 - lon1
a = F.sin(dlat / 2)**2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2)**2
c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
distancia_tramo = R * c

diferencia_altitud = F.col("elevacion") - F.lag("elevacion").over(ventana_global)
desnivel_positivo = F.when(diferencia_altitud > 0, diferencia_altitud).otherwise(0)

# Preparar dataset con métricas tramo a tramo
df_preparado = df_puntos_limpios \
    .withColumn("fecha_diaria", F.to_date("fecha_hora")) \
    .withColumn("dist_km", F.coalesce(distancia_tramo, F.lit(0.0))) \
    .withColumn("desnivel_pos_m", F.coalesce(desnivel_positivo, F.lit(0.0)))

# 3. Agrupar por FECHA DIARIA
df_gold_fechas = df_preparado.groupBy("fecha_diaria").agg(
    F.min("fecha_hora").alias("hora_inicio_jornada"),
    F.max("fecha_hora").alias("hora_fin_jornada"),
    F.round(F.sum("dist_km"), 2).alias("distancia_total_km"),
    F.round(F.min("elevacion"), 1).alias("altitud_min_m"),
    F.round(F.max("elevacion"), 1).alias("altitud_max_m"),
    F.round(F.sum("desnivel_pos_m"), 1).alias("desnivel_acumulado_m"),
    F.count("*").alias("total_puntos_registrados"),
    F.sum(F.when(F.col("es_imputado") == True, 1).otherwise(0)).alias("puntos_reconstruidos")
).orderBy("fecha_diaria")

# 4. Recalcular duraciones y velocidad media refinada por jornada
df_gold_fechas = df_gold_fechas \
    .withColumn("duracion_horas", F.round((F.col("hora_fin_jornada").cast("long") - F.col("hora_inicio_jornada").cast("long")) / 3600, 2)) \
    .withColumn("velocidad_media_kmh", F.round(F.col("distancia_total_km") / F.col("duracion_horas"), 1))

# 5. Sobrescribir la tabla Gold
df_gold_fechas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("gold_rutas_resumen")

# Refrescar catálogo para sincronizar con DirectLake
spark.catalog.refreshTable("gold_rutas_resumen")

print("🏆 ¡Tabla Gold recalculada con éxito sobre los datos limpios!")
display(spark.read.table("gold_rutas_resumen"))

StatementMeta(, e19628f8-5812-484b-8f2b-c6bd368faece, 5, Finished, Available, Finished, False)

🏆 ¡Tabla Gold recalculada con éxito sobre los datos limpios!


SynapseWidget(Synapse.DataFrame, f4bb8ae6-eda0-4098-a869-a150c6bb89bd)

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -------------------------------------------------------------------
# 1. Configura el límite de velocidad mínima (km/h)
# -------------------------------------------------------------------
VELOCIDAD_MINIMA_KMH = 5.0  # Ajusta este valor si lo deseas

# 2. Cargar puntos Silver
df_puntos = spark.read.table("stg_puntos_gpx")

# 3. Definir ventana temporal
ventana_global = Window.orderBy("fecha_hora")

R = 6371.0  # Radio de la Tierra en km

lat1 = F.radians(F.lag("latitud").over(ventana_global))
lon1 = F.radians(F.lag("longitud").over(ventana_global))
lat2 = F.radians(F.col("latitud"))
lon2 = F.radians(F.col("longitud"))

dlat = lat2 - lat1
dlon = lon2 - lon1
a = F.sin(dlat / 2)**2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2)**2
c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
dist_km = R * c

tiempo_seg = F.col("fecha_hora").cast("long") - F.lag("fecha_hora").over(ventana_global).cast("long")
velocidad_calculada = F.when(tiempo_seg > 0, (dist_km / (tiempo_seg / 3600))).otherwise(0.0)

# 4. PASO CLAVE: Crear la columna explícita Y el indicador de primer punto
df_con_vel = df_puntos \
    .withColumn("velocidad_tramo_kmh", F.coalesce(velocidad_calculada, F.lit(0.0))) \
    .withColumn("es_primer_punto", F.lag("latitud").over(ventana_global).isNull())

# 5. AHORA SÍ: Filtrar sobre las columnas ya materializadas
df_silver_filtrado = df_con_vel.filter(
    (F.col("velocidad_tramo_kmh") >= VELOCIDAD_MINIMA_KMH) | (F.col("es_primer_punto") == True)
)

# Estadísticas de la limpieza
puntos_antes = df_puntos.count()
puntos_despues = df_silver_filtrado.count()
puntos_eliminados = puntos_antes - puntos_despues

print(f"📊 Recuento de Limpieza:")
print(f"   - Puntos iniciales: {puntos_antes:,}")
print(f"   - Puntos conservados (>= {VELOCIDAD_MINIMA_KMH} km/h): {puntos_despues:,}")
print(f"   - Puntos eliminados (caminata/parada): {puntos_eliminados:,}")

# 6. Sobrescribir la capa Silver limpia (eliminando las columnas auxiliares)
df_silver_filtrado \
    .drop("velocidad_tramo_kmh", "es_primer_punto") \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("stg_puntos_gpx")

print("✅ Capa Silver actualizada sin los tramos a pie o paradas.")

StatementMeta(, e19628f8-5812-484b-8f2b-c6bd368faece, 4, Finished, Available, Finished, False)

📊 Recuento de Limpieza:
   - Puntos iniciales: 117,745
   - Puntos conservados (>= 5.0 km/h): 115,224
   - Puntos eliminados (caminata/parada): 2,521
✅ Capa Silver actualizada sin los tramos a pie o paradas.


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -------------------------------------------------------------------
# 1. AJUSTA AQUÍ TU FILTRO: Velocidad mínima deseada (en km/h)
# -------------------------------------------------------------------
VELOCIDAD_MINIMA_KMH = 5.0  # Los puntos por debajo de esta velocidad se considerarán caminatas/paradas

# 2. Cargar puntos actuales (Silver)
df_puntos = spark.read.table("stg_puntos_gpx")

# 3. Calcular la velocidad instantánea entre puntos consecutivos para detectar caminatas
ventana_global = Window.orderBy("fecha_hora")

R = 6371.0  # Radio de la Tierra en km

lat1 = F.radians(F.lag("latitud").over(ventana_global))
lon1 = F.radians(F.lag("longitud").over(ventana_global))
lat2 = F.radians(F.col("latitud"))
lon2 = F.radians(F.col("longitud"))

dlat = lat2 - lat1
dlon = lon2 - lon1
a = F.sin(dlat / 2)**2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2)**2
c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
dist_km = R * c

tiempo_seg = F.col("fecha_hora").cast("long") - F.lag("fecha_hora").over(ventana_global).cast("long")
velocidad_tramo_kmh = F.when(tiempo_seg > 0, (dist_km / (tiempo_seg / 3600))).otherwise(0.0)

# Añadir la velocidad estimada del tramo
df_con_vel = df_puntos.withColumn("velocidad_tramo_kmh", F.coalesce(velocidad_tramo_kmh, F.lit(0.0)))

# 4. FILTRAR: Conservar solo los puntos donde se supere la velocidad mínima (o el primer punto)
df_silver_filtrado = df_con_vel.filter(
    (F.col("velocidad_tramo_kmh") >= VELOCIDAD_MINIMA_KMH) | (F.lag("latitud").over(ventana_global).isNull())
)

puntos_antes = df_puntos.count()
puntos_despues = df_silver_filtrado.count()
puntos_eliminados = puntos_antes - puntos_despues

print(f"📊 Recuento de Limpieza:")
print(f"   - Puntos iniciales: {puntos_antes:,}")
print(f"   - Puntos conservados (>= {VELOCIDAD_MINIMA_KMH} km/h): {puntos_despues:,}")
print(f"   - Puntos eliminados (caminata/parada): {puntos_eliminados:,}")

# 5. Sobrescribir la capa Silver limpia
df_silver_filtrado.drop("velocidad_tramo_kmh").write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("stg_puntos_gpx")

StatementMeta(, e19628f8-5812-484b-8f2b-c6bd368faece, 3, Finished, Available, Finished, False)

AnalysisException: It is not allowed to use window functions inside WHERE clause.

In [ ]:
from pyspark.sql import functions as F

# Añadir la columna fecha_diaria a la tabla Silver para relacionarla limpiamente en Power BI
df_silver = spark.read.table("stg_puntos_gpx") \
    .withColumn("fecha_diaria", F.to_date("fecha_hora"))

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("stg_puntos_gpx")

print("✅ Tabla 'stg_puntos_gpx' actualizada con la columna 'fecha_diaria'.")

StatementMeta(, f43f4ae1-cbf8-4265-b05b-761e53faded3, 3, Finished, Available, Finished, False)

✅ Tabla 'stg_puntos_gpx' actualizada con la columna 'fecha_diaria'.


In [ ]:
# Refrescar el catálogo de tablas en Spark
spark.catalog.refreshTable("gold_rutas_resumen")
spark.catalog.refreshTable("stg_puntos_gpx")

print("🔄 Catálogo de tablas sincronizado correctamente.")

StatementMeta(, 1f91ba05-c501-4e49-90ce-0739d59f68d8, 19, Finished, Available, Finished, False)

🔄 Catálogo de tablas sincronizado correctamente.


In [10]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Cargar todos los puntos (reales + imputados)
df_puntos = spark.read.table("stg_puntos_gpx")

# 2. Definir una ÚNICA ventana global ordenada cronológicamente por timestamp
# (Ya no usamos partitionBy("nombre_archivo"))
ventana_global = Window.orderBy("fecha_hora")

# 3. Calcular la distancia y elevación respecto al punto cronológico inmediatamente anterior
R = 6371.0  # Radio de la Tierra en km

lat1 = F.radians(F.lag("latitud").over(ventana_global))
lon1 = F.radians(F.lag("longitud").over(ventana_global))
lat2 = F.radians(F.col("latitud"))
lon2 = F.radians(F.col("longitud"))

dlat = lat2 - lat1
dlon = lon2 - lon1
a = F.sin(dlat / 2)**2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2)**2
c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
distancia_tramo = R * c

diferencia_altitud = F.col("elevacion") - F.lag("elevacion").over(ventana_global)
desnivel_positivo = F.when(diferencia_altitud > 0, diferencia_altitud).otherwise(0)

# Añadir columna de fecha limpia (yyyy-MM-dd) para agrupar por jornada
df_preparado = df_puntos \
    .withColumn("fecha_diaria", F.to_date("fecha_hora")) \
    .withColumn("dist_km", F.coalesce(distancia_tramo, F.lit(0.0))) \
    .withColumn("desnivel_pos_m", F.coalesce(desnivel_positivo, F.lit(0.0)))

# 4. Agrupar EXCLUSIVAMENTE por FECHA DIARIA
df_gold_fechas = df_preparado.groupBy("fecha_diaria").agg(
    F.min("fecha_hora").alias("hora_inicio_jornada"),
    F.max("fecha_hora").alias("hora_fin_jornada"),
    F.round(F.sum("dist_km"), 2).alias("distancia_total_km"),
    F.round(F.min("elevacion"), 1).alias("altitud_min_m"),
    F.round(F.max("elevacion"), 1).alias("altitud_max_m"),
    F.round(F.sum("desnivel_pos_m"), 1).alias("desnivel_acumulado_m"),
    F.count("*").alias("total_puntos_registrados"),
    F.sum(F.when(F.col("es_imputado") == True, 1).otherwise(0)).alias("puntos_reconstruidos")
).orderBy("fecha_diaria")

# 5. Calcular la duración y velocidad media por jornada
df_gold_fechas = df_gold_fechas \
    .withColumn("duracion_horas", F.round((F.col("hora_fin_jornada").cast("long") - F.col("hora_inicio_jornada").cast("long")) / 3600, 2)) \
    .withColumn("velocidad_media_kmh", F.round(F.col("distancia_total_km") / F.col("duracion_horas"), 1))

# 6. Sobrescribir la tabla Gold
df_gold_fechas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("gold_rutas_resumen")

print("🏆 ¡Tabla Gold generada con éxito ordenada y agrupada ÚNICAMENTE por Fecha!")
display(spark.read.table("gold_rutas_resumen"))

StatementMeta(, 1f91ba05-c501-4e49-90ce-0739d59f68d8, 18, Finished, Available, Finished, False)

🏆 ¡Tabla Gold generada con éxito ordenada y agrupada ÚNICAMENTE por Fecha!


SynapseWidget(Synapse.DataFrame, d874aa71-9d8d-42a5-abe2-4454292496b0)

In [9]:
df_comprobacion = spark.read.table("stg_puntos_gpx")

# Ver el desglose exacto de puntos Reales vs Imputados
display(
    df_comprobacion.groupBy("es_imputado")
    .count()
    .withColumnRenamed("count", "total_puntos")
)

StatementMeta(, 1f91ba05-c501-4e49-90ce-0739d59f68d8, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e8ef1be3-0811-4bc0-864a-bebdca7f6896)

In [8]:
# Guardar la tabla Silver permitiendo la actualización de columnas (mergeSchema)
df_final_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("stg_puntos_gpx")

print("💾 ¡Tabla Silver 'stg_puntos_gpx' guardada con éxito con sus 2.651 puntos imputados!")

StatementMeta(, 1f91ba05-c501-4e49-90ce-0739d59f68d8, 16, Finished, Available, Finished, False)

💾 ¡Tabla Silver 'stg_puntos_gpx' guardada con éxito con sus 2.651 puntos imputados!


In [7]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import numpy as np

# 1. Cargar datos originales y definir la ventana de tiempo por archivo
df_base = spark.read.table("stg_puntos_gpx") \
    .withColumn("es_imputado", F.lit(False))

ventana = Window.partitionBy("nombre_archivo").orderBy("fecha_hora")

# 2. Identificar el punto inmediato posterior (LEAD) para detectar los 3 vacíos
df_parejas = df_base \
    .withColumn("fecha_siguiente", F.lead("fecha_hora").over(ventana)) \
    .withColumn("lat_siguiente", F.lead("latitud").over(ventana)) \
    .withColumn("lon_siguiente", F.lead("longitud").over(ventana)) \
    .withColumn("ele_siguiente", F.lead("elevacion").over(ventana))

# Reutilizamos los cálculos de Haversine para filtrar los saltos de >15 km
R = 6371.0
lat1 = F.radians(F.col("latitud"))
lon1 = F.radians(F.col("longitud"))
lat2 = F.radians(F.col("lat_siguiente"))
lon2 = F.radians(F.col("lon_siguiente"))

dlat = lat2 - lat1
dlon = lon2 - lon1
a = F.sin(dlat / 2)**2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2)**2
c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))

df_parejas = df_parejas \
    .withColumn("dist_km", R * c) \
    .withColumn("segundos_gap", F.col("fecha_siguiente").cast("long") - F.col("fecha_hora").cast("long"))

# 3. Filtrar únicamente los puntos de INICIO de los grandes cortes
cortes_para_imputar = df_parejas.filter(
    (F.col("dist_km") > 15.0) & (F.col("segundos_gap") > 120)
).collect()

print(f"🔧 Generando puntos intermedios para los {len(cortes_para_imputar)} cortes detectados...")

# 4. Crear los nuevos puntos interpolados en Python
puntos_interpolados = []

# Fijamos una frecuencia de muestreo para los puntos interpolados (ej. 1 punto cada 10 segundos)
PASO_SEGUNDOS = 10 

for fila in cortes_para_imputar:
    archivo = fila["nombre_archivo"]
    ruta = fila["nombre_ruta"]
    
    t_start = int(fila["fecha_hora"].timestamp())
    t_end = int(fila["fecha_siguiente"].timestamp())
    
    lat_start, lat_end = fila["latitud"], fila["lat_siguiente"]
    lon_start, lon_end = fila["longitud"], fila["lon_siguiente"]
    ele_start, ele_end = fila["elevacion"] or 0.0, fila["ele_siguiente"] or 0.0
    
    # Generar la secuencia de tiempos intermedios
    tiempos_nuevos = list(range(t_start + PASO_SEGUNDOS, t_end, PASO_SEGUNDOS))
    num_puntos = len(tiempos_nuevos)
    
    if num_puntos > 0:
        # Interpolación lineal de latitud, longitud y elevación
        lats = np.linspace(lat_start, lat_end, num_puntos + 2)[1:-1]
        lons = np.linspace(lon_start, lon_end, num_puntos + 2)[1:-1]
        eles = np.linspace(ele_start, ele_end, num_puntos + 2)[1:-1]
        
        for t, lat, lon, ele in zip(tiempos_nuevos, lats, lons, eles):
            puntos_interpolados.append((
                archivo,
                ruta,
                float(lat),
                float(lon),
                float(ele),
                t,  # timestamp en segundos
                True # es_imputado
            ))

print(f"✅ ¡Se han generado {len(puntos_interpolados)} nuevos puntos intermedios para cerrar las rutas!")

# 5. Convertir los puntos generados a DataFrame de PySpark
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, BooleanType

esquema_imp = StructType([
    StructField("nombre_archivo", StringType(), True),
    StructField("nombre_ruta", StringType(), True),
    StructField("latitud", DoubleType(), True),
    StructField("longitud", DoubleType(), True),
    StructField("elevacion", DoubleType(), True),
    StructField("timestamp_sec", LongType(), True),
    StructField("es_imputado", BooleanType(), True)
])

df_imp = spark.createDataFrame(puntos_interpolados, esquema_imp) \
    .withColumn("fecha_hora", F.to_timestamp(F.from_unixtime("timestamp_sec"))) \
    .drop("timestamp_sec")

# 6. Unir los puntos originales con los nuevos puntos interpolados
df_final_silver = df_base.unionByName(df_imp)

# 7. Sobrescribir la Tabla Silver con la ruta completa e impecable
df_final_silver.write.format("delta").mode("overwrite").saveAsTable("stg_puntos_gpx")

print("💾 Tabla Silver 'stg_puntos_gpx' actualizada correctamente con los trayectos cerrados.")

StatementMeta(, 1f91ba05-c501-4e49-90ce-0739d59f68d8, 15, Finished, Available, Finished, False)

🔧 Generando puntos intermedios para los 3 cortes detectados...
✅ ¡Se han generado 2651 nuevos puntos intermedios para cerrar las rutas!


AnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: ad8f5398-7e1d-4f47-8f13-ca842852770a).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- nombre_archivo: string (nullable = true)
-- nombre_ruta: string (nullable = true)
-- latitud: double (nullable = true)
-- longitud: double (nullable = true)
-- elevacion: double (nullable = true)
-- fecha_hora: timestamp (nullable = true)


Data schema:
root
-- nombre_archivo: string (nullable = true)
-- nombre_ruta: string (nullable = true)
-- latitud: double (nullable = true)
-- longitud: double (nullable = true)
-- elevacion: double (nullable = true)
-- fecha_hora: timestamp (nullable = true)
-- es_imputado: boolean (nul

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Cargar la tabla Silver
df = spark.read.table("stg_puntos_gpx")

# 2. Definir la ventana cronológica por archivo
ventana = Window.partitionBy("nombre_archivo").orderBy("fecha_hora")

# 3. Obtener coordenadas y tiempo del punto anterior (LAG)
df_lag = df \
    .withColumn("fecha_prev", F.lag("fecha_hora").over(ventana)) \
    .withColumn("lat_prev", F.lag("latitud").over(ventana)) \
    .withColumn("lon_prev", F.lag("longitud").over(ventana))

# 4. Calcular diferencia de tiempo en minutos
df_lag = df_lag.withColumn(
    "minutos_salto",
    (F.col("fecha_hora").cast("long") - F.col("fecha_prev").cast("long")) / 60
)

# 5. Calcular la distancia en KM usando la fórmula de Haversine en PySpark
# Radio medio de la Tierra = 6371 km
R = 6371.0

lat1_rad = F.radians(F.col("lat_prev"))
lon1_rad = F.radians(F.col("lon_prev"))
lat2_rad = F.radians(F.col("latitud"))
lon2_rad = F.radians(F.col("longitud"))

dlat = lat2_rad - lat1_rad
dlon = lon2_rad - lon1_rad

a = F.sin(dlat / 2)**2 + F.cos(lat1_rad) * F.cos(lat2_rad) * F.sin(dlon / 2)**2
c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))

df_distancia = df_lag.withColumn("distancia_km", R * c)

# 6. FILTRAR NUESTRAS REGLAS DE NEGOCIO:
# - Tiempo mayor a 2 minutos
# - Distancia mayor a 15 kilómetros (ignora paradas en alojamientos)
UMBRAL_DISTANCIA_KM = 15.0
UMBRAL_TIEMPO_MIN = 2.0

cortes_reales = df_distancia.filter(
    (F.col("distancia_km") > UMBRAL_DISTANCIA_KM) & 
    (F.col("minutos_salto") > UMBRAL_TIEMPO_MIN)
).select(
    "nombre_archivo",
    F.col("fecha_prev").alias("inicio_corte"),
    F.col("fecha_hora").alias("fin_corte"),
    F.round("minutos_salto", 1).alias("duracion_minutos"),
    F.round("distancia_km", 2).alias("distancia_salto_km"),
    F.round((F.col("distancia_km") / (F.col("minutos_salto") / 60)), 1).alias("velocidad_implicita_kmh")
).orderBy(F.col("distancia_salto_km").desc())

print(f"🎯 Se han detectado {cortes_reales.count()} saltos de ruta mayores a {UMBRAL_DISTANCIA_KM} km.")
display(cortes_reales)

StatementMeta(, 1f91ba05-c501-4e49-90ce-0739d59f68d8, 14, Finished, Available, Finished, False)

🎯 Se han detectado 3 saltos de ruta mayores a 15.0 km.


SynapseWidget(Synapse.DataFrame, 6bb67771-62b4-4bf1-9002-e851f12ee581)

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Definir la ventana para ordenar los puntos cronológicamente dentro de cada archivo
ventana_ruta = Window.partitionBy("nombre_archivo").orderBy("fecha_hora")

# 2. Obtener la fecha/hora, latitud y longitud del punto ANTERIOR
df_saltos = spark.read.table("stg_puntos_gpx") \
    .withColumn("fecha_hora_prev", F.lag("fecha_hora").over(ventana_ruta)) \
    .withColumn("lat_prev", F.lag("latitud").over(ventana_ruta)) \
    .withColumn("lon_prev", F.lag("longitud").over(ventana_ruta))

# 3. Calcular la diferencia en segundos con el punto anterior
df_saltos = df_saltos.withColumn(
    "diferencia_segundos",
    F.col("fecha_hora").cast("long") - F.col("fecha_hora_prev").cast("long")
)

# 4. Filtrar cortes significativos (por ejemplo, saltos de más de 60 segundos)
# ¡Puedes ajustar este valor de 60 si tus lecturas normales eran cada segundo!
UMBRAL_CORTE_SEGUNDOS = 60

cortes_detectados = df_saltos.filter(F.col("diferencia_segundos") > UMBRAL_CORTE_SEGUNDOS) \
    .select(
        "nombre_archivo",
        "fecha_hora_prev",
        "fecha_hora",
        F.round(F.col("diferencia_segundos") / 60, 2).alias("minutos_sin_cobertura")
    ) \
    .orderBy(F.col("minutos_sin_cobertura").desc())

print(f"⚠️ Se han detectado {cortes_detectados.count()} cortes mayores a {UMBRAL_CORTE_SEGUNDOS} segundos.")
display(cortes_detectados)

StatementMeta(, 1f91ba05-c501-4e49-90ce-0739d59f68d8, 13, Finished, Available, Finished, False)

⚠️ Se han detectado 855 cortes mayores a 60 segundos.


SynapseWidget(Synapse.DataFrame, 3ba11344-b417-4077-99e1-15afb5cb0199)

In [4]:
# Cargar la tabla Delta
df = spark.read.table("stg_puntos_gpx")

# 1. Contar el TOTAL REAL de puntos GPS guardados
total_filas = df.count()
print(f"📍 Total de puntos GPS registrados en la tabla: {total_filas}")

print("-" * 50)

# 2. Ver cuántos puntos aporta CADA archivo GPX
display(df.groupBy("nombre_archivo").count().withColumnRenamed("count", "puntos_gps"))

StatementMeta(, 1f91ba05-c501-4e49-90ce-0739d59f68d8, 12, Finished, Available, Finished, False)

📍 Total de puntos GPS registrados en la tabla: 115094
--------------------------------------------------


SynapseWidget(Synapse.DataFrame, be1e0e92-e07b-458c-ba2b-59443c4996b5)

In [3]:
import os
import gpxpy
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# 1. Definir la ruta donde subiste los archivos en la capa Bronze
# Recordatorio: en Fabric la carpeta 'Files' se mapea localmente en /lakehouse/default/Files/
ruta_gpx = "/lakehouse/default/Files/gpx/"

datos_rutas = []

# 2. Recorrer todos los archivos .gpx de la carpeta
for archivo in os.listdir(ruta_gpx):
    if archivo.endswith(".gpx"):
        path_completo = os.path.join(ruta_gpx, archivo)
        
        with open(path_completo, 'r') as f:
            gpx = gpxpy.parse(f)
            
            # Extraer los puntos GPS de cada track y segmento
            for track in gpx.tracks:
                for segment in track.segments:
                    for point in segment.points:
                        datos_rutas.append((
                            archivo,
                            track.name or archivo,
                            point.latitude,
                            point.longitude,
                            point.elevation,
                            point.time
                        ))

# 3. Definir el esquema (tipos de datos) de nuestra tabla
esquema = StructType([
    StructField("nombre_archivo", StringType(), True),
    StructField("nombre_ruta", StringType(), True),
    StructField("latitud", DoubleType(), True),
    StructField("longitud", DoubleType(), True),
    StructField("elevacion", DoubleType(), True),
    StructField("fecha_hora", TimestampType(), True)
])

# 4. Crear el DataFrame de PySpark
df_rutas = spark.createDataFrame(datos_rutas, esquema)

# 5. Guardar los datos en la Capa Silver como una Tabla Delta llamada 'stg_puntos_gpx'
df_rutas.write.format("delta").mode("overwrite").saveAsTable("stg_puntos_gpx")

# Mostrar una vista previa de las primeras 10 filas
display(df_rutas.limit(10))

StatementMeta(, 1f91ba05-c501-4e49-90ce-0739d59f68d8, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d4f47122-78e5-487a-b18a-294c191b5743)

In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
%pip install gpxpy

StatementMeta(, 1f91ba05-c501-4e49-90ce-0739d59f68d8, 8, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 1.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

